# Lookup: computed once per block, written to disk, read back

Two synthetic blocks and three zones that straddle them. First the zone lookup, the zone id
of every pixel; then the zone cell weights, a population raster summed onto pairs of zone and
coarse cell, which turn a field on a coarse grid into sums per zone without touching the
pixels again. Everything is written to a folder beside this notebook and removed at the end;
real builds go to the output node, see `docs/lookup.md`.

In [ ]:
import shutil
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from rasterio.windows import Window
from shapely.geometry import box

from idd_spatial_analysis.constants import WGS84, WORLD_CYLINDRICAL_EQUAL_AREA
from idd_spatial_analysis.frame import block_grid, blocks_intersecting, grid_from_coords
from idd_spatial_analysis.lookup import (
    build_zone_cell_weights_block,
    build_zone_lookup_block,
    collect_zone_cell_weights,
    collect_zone_lookup,
    load_zone_cell_weights,
    load_zone_lookup_index,
    load_zone_lookup_metadata,
    read_zone_lookup_block,
)
from idd_spatial_analysis.raster import NearestIndexMap, rasterize_zones
from idd_spatial_analysis.zonal import ZoneCellPairs, collate_blocks, weighted_sums

WORK = Path.cwd() / "_lookup_vignette_tmp"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

In [ ]:
# A tile index of two 6.4 km blocks side by side, and three zones over both of them.
tiles = gpd.GeoDataFrame(
    {"block_key": ["B-0000X-0000Y", "B-0001X-0000Y"]},
    geometry=[box(0, 0, 6400, 6400), box(6400, 0, 12800, 6400)],
    crs=WORLD_CYLINDRICAL_EQUAL_AREA,
)
zones = gpd.GeoDataFrame(
    {"zone": np.array([1, 2, 3], dtype="int32")},
    geometry=[box(0, 3200, 9600, 6400), box(9600, 3200, 12800, 6400), box(0, 0, 12800, 3200)],
    crs=WORLD_CYLINDRICAL_EQUAL_AREA,
)
# Build: one raster and one small table per block. Blocks are independent of each other.
keys = blocks_intersecting(tiles, zones)
grids = {key: block_grid(tiles, key, resolution=100.0) for key in keys}
for key, grid in grids.items():
    print(key)
    print(build_zone_lookup_block(zones, "zone", grid, WORK, key))
print(sorted(str(path.relative_to(WORK)) for path in WORK.rglob("*") if path.is_file()))

In [ ]:
# Collect once every block is built: the index, the per-zone summary and the sidecar.
summary = collect_zone_lookup(WORK, keys, zones["zone"].tolist(), zone_set="vignette")
print(summary)
index = load_zone_lookup_index(WORK)
print(index)
print(load_zone_lookup_metadata(WORK))

In [ ]:
# Read a block back, whole or just the window the index gives for one zone.
mask, grid = read_zone_lookup_block(WORK, "B-0001X-0000Y")
print(mask.dtype, mask.shape, np.unique(mask))
row = index.loc[(index["block_key"] == "B-0001X-0000Y") & (index["zone"] == 2)].iloc[0]
window = Window(row["col_off"], row["row_off"], row["width"], row["height"])
part, part_grid = read_zone_lookup_block(WORK, "B-0001X-0000Y", window=window)
print(part.shape, np.unique(part), part_grid.bounds)

In [ ]:
# Population-weighted means per zone from the stored masks: no polygons at this point.
# The sums of each block add up, so the divide happens once at the end.
rng = np.random.default_rng(0)
partials, again = [], []
for key, grid in grids.items():
    population = rng.gamma(2.0, 50.0, size=grid.shape).astype("float32")
    temperature = (25 + 5 * rng.standard_normal(grid.shape)).astype("float32")
    stored, _ = read_zone_lookup_block(WORK, key)
    partials.append(weighted_sums(stored, population, temperature))
    fresh, _ = rasterize_zones(zones, "zone", grid)
    again.append(weighted_sums(fresh, population, temperature))
sums = collate_blocks(partials)
sums["mean"] = sums["weighted_sum"] / sums["weight_sum"]
print(sums)
print(
    "same as rasterizing again:",
    sums[["weighted_sum", "weight_sum"]].equals(
        collate_blocks(again)[["weighted_sum", "weight_sum"]]
    ),
)

## Zone cell weights

A field on a coarse grid (here 0.01 degree cells in lon/lat) gives every pixel the value of
the cell its centre falls in. The weighted sum over a zone's pixels is then a sum over cells
of the cell's value times the zone's population inside that cell. Those populations per
(zone, cell) pair are computed once per block and population year, collected per year, and
applied to any number of fields.

In [ ]:
# The coarse grid from cell-centre coordinates; the registration is stated, never assumed.
lon = 0.005 + 0.01 * np.arange(12)
lat = 0.055 - 0.01 * np.arange(6)
coarse = grid_from_coords(lon, lat, WGS84, registration="center")

WEIGHTS = WORK / "zone_cell_weights"
years = ["2000", "2001"]
populations = {
    year: {
        key: rng.gamma(2.0, 50.0, size=grid.shape).astype("float32") for key, grid in grids.items()
    }
    for year in years
}
# Round 1, per block: pair the zoned pixels with coarse cells once, then one table per year.
for key, grid in grids.items():
    stored, _ = read_zone_lookup_block(WORK, key)
    pairs = ZoneCellPairs.build(stored, grid, coarse)
    print(key, len(pairs), "pairs; pixel centres transformed by:", pairs.centre_transform)
    for year in years:
        build_zone_cell_weights_block(pairs, populations[year][key], WEIGHTS, year, key)
# Round 2, per year: add the blocks up. A zone that straddles the blocks can share a cell.
for year in years:
    collected = collect_zone_cell_weights(WEIGHTS, year, keys, zone_set="vignette")
    print(year, len(collected), "pairs,", round(collected["weight"].sum()), "people")
print(sorted(str(path.relative_to(WEIGHTS)) for path in WEIGHTS.rglob("*") if path.is_file()))

In [ ]:
# Apply: any field on the coarse grid, without the pixels. The table carries its grid.
weights = load_zone_cell_weights(WEIGHTS, "2000")
print(weights.table.head())
print(weights.coarse)
field = (25 + 5 * rng.standard_normal(coarse.shape)).astype("float32")
sums = weights.apply(field, coarse)
sums["mean"] = sums["weighted_sum"] / sums["weight_sum"]
print(sums)

# The same sums the long way: put the field on the pixels of each block, then weighted_sums.
pixel_pass = collate_blocks(
    [
        weighted_sums(
            read_zone_lookup_block(WORK, key)[0],
            populations["2000"][key],
            NearestIndexMap.build(coarse, grid).apply(field),
        )
        for key, grid in grids.items()
    ]
)
print("same as the pixel pass:", np.allclose(sums["weighted_sum"], pixel_pass["weighted_sum"]))

# Reading the same coordinates as cell edges describes a grid half a cell away: refused.
half_a_cell_off = grid_from_coords(lon, lat, WGS84, registration="lower_left")
try:
    weights.apply(field, half_a_cell_off)
except ValueError as error:
    print("refused:", error)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for axis, key in zip(axes, keys, strict=True):
    stored, grid = read_zone_lookup_block(WORK, key)
    left, bottom, right, top = grid.bounds
    axis.imshow(stored, extent=(left, right, bottom, top), vmin=0, vmax=3)
    axis.set_title(key)
plt.show()
shutil.rmtree(WORK)